In [1]:
import os
import sys
import shutil
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import pandas as pd
import numpy as np
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from torch.utils.data import DataLoader
import utils.file_IO as file_IO
import pandas as pd

In [3]:
kind='patches_224' 
input_preprocessed=file_IO.load_preprocessed_files(kind, mode='contrastive')
train_df = pd.read_csv(f"{source_path}\\outputs\\preprocessed_data\\{input_preprocessed}")
train_df['file_name'].iloc[0]

'C:\\Users\\andre\\PhD\\Datasets\\rimes\\images_blocs_de_texte\\images_blocs_de_texte\\DVD1\\00035_L.jpg'

In [30]:
modality='train'
input_preprocessed=file_IO.load_preprocessed_files(kind='patches_224', mode=modality)
train_df = pd.read_csv(f"{source_path}\\outputs\\preprocessed_data\\{input_preprocessed}")
print(train_df['writer'].nunique(), 'unique writers')
train_df['page'] = train_df.groupby(['writer', 'isEng', 'same_text']).ngroup()
print(len(train_df)/train_df['page'].nunique(), 'samples per writer')
num_female_writers = train_df.loc[train_df['male'] == 0, 'writer'].nunique()
print(f"Number of writers for which isMale==0: {num_female_writers}",num_female_writers/train_df['writer'].nunique())

282 unique writers
50.0 samples per writer
Number of writers for which isMale==0: 143 0.5070921985815603


In [20]:
modality='val'
input_preprocessed=file_IO.load_preprocessed_files(kind='patches_224', mode=modality)
val_df = pd.read_csv(f"{source_path}\\outputs\\preprocessed_data\\{input_preprocessed}")
modality='test'
input_preprocessed=file_IO.load_preprocessed_files(kind='patches_224', mode=modality)
test_df = pd.read_csv(f"{source_path}\\outputs\\preprocessed_data\\{input_preprocessed}")
writers_val = set(val_df['writer'].unique())
writers_test = set(test_df['writer'].unique())
common_writers = writers_val & writers_test
print(f"Number of unique writers in val_df: {len(writers_val)}")
print(f"Number of unique writers in test_df: {len(writers_test)}")
print(f"Number of writers present in both: {len(common_writers)}")
if len(common_writers) == 0:
    print("The sets of writers are disjoint.")
else:
    print(f"Common writer IDs: {sorted(common_writers)}")

Number of unique writers in val_df: 71
Number of unique writers in test_df: 122
Number of writers present in both: 0
The sets of writers are disjoint.


In [ ]:
#train 1-282, 50 patches, 50% women
#val 71, 50 patches, 60% women
#test 122 50 patches 56% women

In [17]:
input_filename='icdar_test_private_df_20250716_101521.csv'
#'icdar_test_public_df_20250716_101521.csv'#'icdar_test_private_df_20250716_101521.csv'#'icdar_train_df_patches_20250515_164130.csv'
running = 'new-laptop'
saved = 'new-laptop'
train_df = pd.read_csv(f"{source_path}\\outputs\\preprocessed_data\\{input_filename}")
print(train_df['writer'].max(),train_df['writer'].min(),train_df['writer'].max()-train_df['writer'].min())
train_df['page'] = train_df.groupby(['writer', 'isEng', 'same_text']).ngroup()
print(len(train_df)/train_df['page'].nunique(), 'samples per writer')

474 283 191
1.0 samples per writer


# analysis of val, test files

In [26]:
selected_FE ='clip-vit-large-patch14'# 'clip-vit-large-patch14'#'DeiT-Tiny' #'resnet18'#'clip-vit-large-patch14'  # Example feature extractor, can be changed
input_dir=source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_FE}\\representation_extraction\\extracted_representation'
use_test_private = False
data_augmentation = False
suffix = '_augmented' if data_augmentation else ''
kind = 'patches_224'  # Example kind, can be changed
train_filename,val_filename, test_filename = file_IO.load_input_files(source_path,selected_FE,kind,suffix)
if use_test_private:
    val_df = pd.read_csv(test_filename)
else:
    val_df = pd.read_csv(val_filename) 
val_df['train'] = 0  # Mark validation data
print(val_df['writer'].nunique(), 'unique writers in val_df')
print(val_df['writer'].max(),val_df['writer'].min(),val_df['writer'].max()-val_df['writer'].min())
print(len(val_df)/val_df['page'].nunique(), 'samples per writer')
train_df = pd.read_csv(train_filename)
print(train_df['writer'].nunique(), 'unique writers in train_df')
print(train_df['writer'].max(),train_df['writer'].min(),train_df['writer'].max()-train_df['writer'].min())
print(len(train_df)/train_df['page'].nunique(), 'samples per writer')


71 unique writers in val_df
475 284 191
20.0 samples per writer
282 unique writers in train_df
282 1 281
40.0 samples per writer


In [ ]:
#clip-vit
#train 282-1 40 patches, val 71 20 patches, test 122 40 patches

#resnet50
#train, val 284-475 20 patches, test 283-474 40 patches

#deit-tiny
#train, val 284-475 40 patches, test 283-474 40 patches

#resnet18
#train, val 284-475 40 patches, test 283-474 40 patches

# grid search analysis

In [6]:
selected_FE = 'resnet50'
search_type = 'grid_search'#'grid_search'  # or 'random_search'
model_list = ['MLPClassifier1','MLPClassifier2']
best_result = [np.inf, np.inf]
save_common=source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_FE}\\representation_extraction\\torch_model_trained_on_rep\\'
prev_log = os.path.join(save_common, f'grid_search_results.csv')
if os.path.exists(prev_log):
    grid_results = pd.read_csv(prev_log) 
    grid_results['type'] = 'grid_search'
prev_log = os.path.join(save_common, f'random_search_results.csv')
if os.path.exists(prev_log):
    random_results = pd.read_csv(prev_log)  
    random_results['type'] = 'random_search'
prev_results = grid_results.copy()
#prev_results = pd.concat([grid_results, random_results], ignore_index=True)
prev_results.sort_values(by='best_val_loss', inplace=True)
print(prev_results.columns)
cols_to_show = ['dropout','n_neurons','lr','best_val_loss','best_val_acc','optimizer','log_grad_norm','best_epoch','with_input_norm','model_name']
display(prev_results[cols_to_show].head())
prev_results.sort_values(by='best_val_acc', inplace=True, ascending=False)
display(prev_results[cols_to_show].head(15))

Index(['best_val_loss', 'best_val_acc', 'best_epoch', 'best_train_loss',
       'best_train_acc', 'last_epoch', 'last_val_loss', 'last_val_acc',
       'last_train_loss', 'last_train_acc', 'lr', 'dropout', 'n_neurons',
       'model_name', 'optimizer', 'scheduler', 'log_grad_norm', 'activation',
       'with_input_norm', 'id', 'type'],
      dtype='object')


,dropout,n_neurons,lr,best_val_loss,best_val_acc,optimizer,log_grad_norm,best_epoch,with_input_norm,model_name
553,0.7,32,0.0010,0.578611,0.670599,Adam,True,2,dataset_norm,MLPClassifier1
152,0.4,64,0.0010,0.578629,0.683451,AdamW,False,0,dataset_norm,MLPClassifier1
487,0.7,64,0.0010,0.578672,0.669542,Adam,False,7,batch_norm,MLPClassifier2
452,0.7,64,0.0010,0.579346,0.678521,AdamW,False,8,batch_norm,MLPClassifier2
559,0.4,256,0.0001,0.579510,0.686972,AdamW,True,1,dataset_norm,MLPClassifier1


,dropout,n_neurons,lr,best_val_loss,best_val_acc,optimizer,log_grad_norm,best_epoch,with_input_norm,model_name
64,0.4,128,0.0010,0.598137,0.691373,AdamW,True,1,dataset_norm,MLPClassifier1
364,0.2,512,0.0001,0.614268,0.689789,AdamW,False,1,batch_norm,MLPClassifier1
401,0.2,32,0.0100,0.587057,0.688908,Adam,True,0,dataset_norm,MLPClassifier2
559,0.4,256,0.0001,0.579510,0.686972,AdamW,True,1,dataset_norm,MLPClassifier1
413,0.4,32,0.0010,0.581908,0.686972,AdamW,True,2,dataset_norm,MLPClassifier2
528,0.4,512,0.0010,0.619997,0.685387,Adam,False,1,batch_norm,MLPClassifier1
198,0.7,512,0.0001,0.591272,0.685211,AdamW,True,2,dataset_norm,MLPClassifier1
134,0.7,128,0.0010,0.602400,0.685211,Adam,True,4,dataset_norm,MLPClassifier1
48,0.4,64,0.0001,0.588323,0.683979,Adam,False,4,batch_norm,MLPClassifier2
8,0.7,128,0.0001,0.585923,0.683979,AdamW,True,5,batch_norm,MLPClassifier1


In [15]:
display(prev_results[prev_results['type'] == 'random_search'].sort_values('best_val_loss').head(10))

,best_val_loss,best_val_acc,best_epoch,best_train_loss,best_train_acc,last_epoch,last_val_loss,last_val_acc,last_train_loss,last_train_acc,lr,dropout,n_neurons,model_name,optimizer,scheduler,log_grad_norm,activation,id,type
146,0.547029,0.709507,10,0.520723,0.733067,25,0.602948,0.690669,0.455974,0.772606,0.0010,0.4,256,MLPClassifier1,Adam,OneCycleLR,True,relu,50,random_search
132,0.548826,0.703345,25,0.606372,0.665492,40,0.560320,0.686796,0.594957,0.678280,0.0001,0.9,32,MLPClassifier2,Adam,CyclicalLR,False,relu,36,random_search
100,0.549212,0.705634,18,0.599334,0.672451,33,0.560364,0.695246,0.594653,0.678613,0.1000,0.2,32,MLPClassifier1,SGD,StepLR,True,relu,4,random_search
158,0.549562,0.700352,18,0.554518,0.712389,33,0.574515,0.698944,0.535752,0.724091,0.0100,0.1,32,MLPClassifier1,SGD,no_scheduling,False,relu,62,random_search
192,0.550859,0.704225,59,0.570212,0.696786,74,0.584685,0.686444,0.561667,0.703391,0.0100,0.7,512,MLPClassifier2,SGD,no_scheduling,False,leaky_relu,96,random_search
189,0.552355,0.707923,43,0.545379,0.722562,58,0.554189,0.705458,0.530024,0.737079,0.0001,0.9,128,MLPClassifier2,Adam,CosineAnnealingWarmRestarts,False,relu,93,random_search
128,0.552921,0.703169,27,0.554016,0.714938,42,0.572642,0.700000,0.547461,0.720168,0.0100,0.4,128,MLPClassifier1,SGD,no_scheduling,True,tanh,32,random_search
107,0.552981,0.700176,74,0.584391,0.689317,89,0.564790,0.694894,0.561355,0.709818,0.0100,0.1,256,MLPClassifier2,SGD,CosineAnnealingWarmRestarts,True,relu,11,random_search
186,0.552989,0.701937,44,0.608923,0.677859,59,0.570926,0.700352,0.552536,0.717176,0.1000,0.9,512,MLPClassifier1,SGD,CosineAnnealingWarmRestarts,False,tanh,90,random_search
153,0.553047,0.701761,17,0.549216,0.719947,32,0.565334,0.701761,0.543553,0.719371,0.0001,0.9,128,MLPClassifier2,Adam,CyclicalLR,False,tanh,57,random_search


# are preprocessed files ok?! Yes

In [3]:
source_file_train='icdar_train_df_patches_20250716_113702'
extra_file_train = 'icdar_train_df_body_20250523_181312'
train_df = pd.read_csv(f"{source_path}\\outputs\\preprocessed_data\\{source_file_train}.csv")
train_df_2 = pd.read_csv(f"{source_path}\\outputs\\preprocessed_data\\{extra_file_train}.csv")

In [4]:
cols_to_drop = [c for c in train_df.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
print(f"Columns to drop: {cols_to_drop}")

Columns to drop: ['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index', 'x', 'y', 'x2', 'y2', 'n_cc', 'black_ratio']


In [5]:
cols_to_drop = [c for c in train_df_2.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
print(f"Columns to drop: {cols_to_drop}")

Columns to drop: ['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index', 'x', 'y', 'x2', 'y2']


In [7]:
train_df_2['file_name'][0]

'C:\\Users\\andre\\PhD\\Datasets\\ICDAR 2013 - Gender Identification Competition Dataset\\unzipped\\1_50\\0001_1.jpg'

In [10]:
train_df=file_IO.change_filename_from_to(train_df, fr='old-laptop', to='new-laptop')

In [11]:
train_df['file_name'][0]

'C:\\Users\\andre\\PhD\\Datasets\\ICDAR 2013 - Gender Identification Competition Dataset\\unzipped\\1_50\\0001_1.jpg'

In [12]:
combined = pd.concat([train_df[['writer', 'isEng', 'same_text']],
                            train_df_2[['writer', 'isEng', 'same_text']]]).drop_duplicates()
# Create consistent group ids
combined['page'] = combined.groupby(['writer', 'isEng', 'same_text']).ngroup()
print(len(combined))

1128


In [13]:
# Merge group ids back to each original DataFrame
train_df = train_df.merge(combined, on=['writer', 'isEng', 'same_text'], how='left')
train_df_2 = train_df_2.merge(combined, on=['writer', 'isEng', 'same_text'], how='left')

In [14]:
cols_to_drop = [c for c in train_df.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
train_df[cols_to_drop].columns

Index(['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index',
       'x', 'y', 'x2', 'y2', 'n_cc', 'black_ratio', 'page'],
      dtype='object')

In [15]:
cols_to_drop = [c for c in train_df_2.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
train_df_2[cols_to_drop].columns

Index(['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index',
       'x', 'y', 'x2', 'y2', 'page'],
      dtype='object')

# are extracted files ok?! yes, error was in the page column being already present

In [3]:
train=r"c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\online_deep_feature_extraction\clip-vit-large-patch14\representation_extraction\extracted_representation\train\clip-vit-large-patch14_features_icdar_train_df_patches_20250716_113702.csv"
extra=r"c:\Users\andre\VsCode\PD related projects\gender_detection\outputs\online_deep_feature_extraction\clip-vit-large-patch14\representation_extraction\extracted_representation\extra_view\train\clip-vit-large-patch14_features_icdar_train_df_body_20250523_181312.csv"

In [25]:
train_df = pd.read_csv(train)
train_df_2 = pd.read_csv(extra)

In [26]:
cols_to_drop = [c for c in train_df.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
print(f"Columns to drop: {cols_to_drop}")
cols_to_drop = [c for c in train_df_2.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
print(f"Columns to drop: {cols_to_drop}")

Columns to drop: ['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index', 'x', 'y', 'x2', 'y2', 'n_cc', 'black_ratio', 'page']
Columns to drop: ['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index', 'x', 'y', 'x2', 'y2', 'page']


In [27]:
train_df.drop(columns=['page'], inplace=True)
train_df_2.drop(columns=['page'], inplace=True)

In [28]:
combined = pd.concat([train_df[['writer', 'isEng', 'same_text']],
                            train_df_2[['writer', 'isEng', 'same_text']]]).drop_duplicates()
# Create consistent group ids
combined['page'] = combined.groupby(['writer', 'isEng', 'same_text']).ngroup()
print(len(combined))
# Merge group ids back to each original DataFrame
train_df = train_df.merge(combined, on=['writer', 'isEng', 'same_text'], how='left')
train_df_2 = train_df_2.merge(combined, on=['writer', 'isEng', 'same_text'], how='left')
cols_to_drop = [c for c in train_df.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
print(train_df[cols_to_drop].columns)
cols_to_drop = [c for c in train_df_2.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
print(train_df_2[cols_to_drop].columns)

1128
Index(['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index',
       'x', 'y', 'x2', 'y2', 'n_cc', 'black_ratio', 'page'],
      dtype='object')
Index(['writer', 'isEng', 'same_text', 'file_name', 'male', 'train', 'index',
       'x', 'y', 'x2', 'y2', 'page'],
      dtype='object')


# is the merging ok? cause i have troubles in the ensemble vs individual accuracy

In [34]:
train_1 = pd.read_csv(train)
train_2 = pd.read_csv(extra)
train_1.drop(columns=['page'], inplace=True) if 'page' in train_1.columns else None
train_2.drop(columns=['page'], inplace=True) if 'page' in train_2.columns else None
# Concatenate both datasets to build a unified group mapping
combined = pd.concat([train_1[['writer', 'isEng', 'same_text']],
                    train_2[['writer', 'isEng', 'same_text']]]).drop_duplicates()
# Create consistent group ids
combined['page'] = combined.groupby(['writer', 'isEng', 'same_text']).ngroup()
# Merge group ids back to each original DataFrame
train_1 = train_1.merge(combined, on=['writer', 'isEng', 'same_text'], how='left')
train_2 = train_2.merge(combined, on=['writer', 'isEng', 'same_text'], how='left')
cols_to_drop_1 = [c for c in train_1.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
cols_to_drop_2 = [c for c in train_2.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
#print(train_2[cols_to_drop_2].head())
common_cols = list(set(cols_to_drop_1) & set(cols_to_drop_2))
# Remove 'page' from common_cols if present
if 'page' in common_cols:
    common_cols.remove('page')
    common_cols.remove('file_name')  # Assuming 'file_name' is not needed for merging

In [35]:
train_2.drop(columns=common_cols, inplace=True, errors='ignore')
num_pages = combined['page'].nunique()
patch_1_per_page = int(len(train_1) / num_pages)
patch_2_per_page = int(len(train_2) / num_pages)
# Repeat train_2 so it matches the number of patches per page in train_1
repeat_factor = patch_1_per_page // patch_2_per_page
if repeat_factor > 1:
    train_2 = pd.concat([train_2] * repeat_factor, ignore_index=True)

In [29]:
print(len(train_1),len(train_2))
print(patch_1_per_page, patch_2_per_page)

45120 1128
40 1


In [36]:
# Add a 'patch_num' column to train_1: unique number per row within each 'page' group
train_1['patch_num'] = train_1.groupby('page').cumcount()
train_2['patch_num'] = train_2.groupby('page').cumcount()
train_1['patch_num'] = train_1['patch_num'] % patch_2_per_page
merged_df = pd.merge(train_1, train_2, on=['page','patch_num'], suffixes=('_1', '_2'))

In [37]:
print(len(merged_df))
cols_to_drop = [c for c in merged_df.columns if not(c.startswith('f') and len(c) > 1 and c[1].isdigit())]
print(f"Columns to drop: {cols_to_drop}")

45120
Columns to drop: ['writer', 'isEng', 'same_text', 'file_name_1', 'male', 'train', 'index', 'x', 'y', 'x2', 'y2', 'n_cc', 'black_ratio', 'page', 'patch_num', 'file_name_2']


In [38]:
if 'file_name_1' in merged_df.columns and 'file_name_2' in merged_df.columns:
    all_equal = (merged_df['file_name_1'] == merged_df['file_name_2']).all()
    print(f"file_name_1 always equals file_name_2: {all_equal}")
else:
    print("file_name_1 or file_name_2 not found in merged_df columns.")

file_name_1 always equals file_name_2: True


In [39]:
# For each group of rows with the same 'page', check if all 'file_name_1' values are equal
file_name_consistency = merged_df.groupby('page')['file_name_1'].nunique()
all_equal = (file_name_consistency == 1).all()
print(f"file_name_1 is always equal within each page group: {all_equal}")
if not all_equal:
    inconsistent_pages = file_name_consistency[file_name_consistency > 1].index.tolist()
    print(f"Inconsistent pages: {inconsistent_pages}")

file_name_1 is always equal within each page group: True


In [41]:
# For each group of rows with the same 'page', check if all 'file_name_1' values are equal
file_name_consistency = merged_df.groupby('page')['male'].nunique()
all_equal = (file_name_consistency == 1).all()
print(f"male is always equal within each page group: {all_equal}")
if not all_equal:
    inconsistent_pages = file_name_consistency[file_name_consistency > 1].index.tolist()
    print(f"Inconsistent pages: {inconsistent_pages}")

male is always equal within each page group: True


In [20]:
# Get the columns to drop (non-feature columns) for train_2
cols_to_keep = [c for c in train_2.columns if c.startswith('f') and len(c) > 1 and c[1].isdigit()]

# Select the subset where page == 0
subset = train_2[train_2['page'] == 0]

# Check if all values in each of these columns are the same
B=True
for col in cols_to_keep:
    unique_vals = subset[col].unique()
    if len(unique_vals) > 1:
        B=False
        print(f"Column {col} has multiple unique values: {unique_vals}")
    else:
        pass
    #print(f"{col}: {unique_vals} (n_unique={len(unique_vals)})")